In [1]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit, Aer, transpile, assemble, execute
from qiskit.visualization import plot_histogram
from math import gcd
from numpy.random import randint
import pandas as pd
from fractions import Fraction
print("Imports Successful")

Imports Successful


In [2]:
from qiskit.aqua.algorithms import Shor
from qiskit.aqua import QuantumInstance

# FINDING FACTORS OF 15 USING QUANTUM SIMULATOR

In [23]:
N = 15
np.random.seed(1) # This is to ensure we get 7 as the random number
a = randint(2, 15)
n_count = 8  # number of counting qubits
print(a)

7


In [24]:
# modular exponentiation function
def c_amod15(a, power):
    """Controlled multiplication by a mod 15"""
    if a not in [2,7,8,11,13]:
        raise ValueError("'a' must be 2,7,8,11 or 13")
    U = QuantumCircuit(4)        
    for iteration in range(power):
            U.swap(2,3)
            U.swap(1,2)
            U.swap(0,1)
            for q in range(4):
                U.x(q)
    U = U.to_gate()
    U.name = "%i^%i mod 15" % (a, power)
    c_U = U.control()
    return c_U

In [25]:
# Quantum Fourier Transform circuit
def qft_dagger(n):    
    qc = QuantumCircuit(n)
    for qubit in range(n//2):
        qc.swap(qubit, n-qubit-1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi/float(2**(j-m)), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc

In [26]:
# Creating QuantumCircuit with 8 counting qubits plus 4 qubits for U to act on
def qpe_amod15(a):
    n_count = 8
    qc = QuantumCircuit(4+n_count, n_count)
    for q in range(n_count):
        qc.h(q)     
    qc.x(3+n_count) 
    for q in range(n_count):
        qc.append(c_amod15(a, 2**q), 
                 [q] + [i+n_count for i in range(4)])
    qc.append(qft_dagger(n_count), range(n_count)) 
    qc.measure(range(n_count), range(n_count))  
    qasm_sim = Aer.get_backend('qasm_simulator')
    result=execute(qc,backend=qasm_sim,shots=1000, memory=True).result()
    readings = result.get_memory()
    print("Register Reading: " + readings[0])
    phase = int(readings[0],2)/(2**n_count)
    print("Corresponding Phase: %f" % phase)
    return phase


In [27]:
# Using the quantum ciruit above with a=7  to factor 15
factor_found = False
attempt = 0
while not factor_found:
    attempt += 1
    print("\nAttempt %i:" % attempt)
    phase = qpe_amod15(a) # Phase = s/r
    frac = Fraction(phase).limit_denominator(N) # Denominator should (hopefully!) tell us r
    r = frac.denominator
    print("Result: r = %i" % r)
    if phase != 0:
        # Guesses for factors are gcd(x^{r/2} ±1 , 15)
        guesses = [gcd(a**(r//2)-1, N), gcd(a**(r//2)+1, N)]
        print("Guessed Factors: %i and %i" % (guesses[0], guesses[1]))
        for guess in guesses:
            if guess not in [1,N] and (N % guess) == 0: # Check to see if guess is a factor
                print("*** Non-trivial factor found: %i ***" % guess)
                factor_found = True


Attempt 1:
Register Reading: 11000000
Corresponding Phase: 0.750000
Result: r = 4
Guessed Factors: 3 and 5
*** Non-trivial factor found: 3 ***
*** Non-trivial factor found: 5 ***


# USING QUANTUM COMPUTER

# FINDING FACTORS OF 15

In [28]:
backend=Aer.get_backend('qasm_simulator')
quantum_instance=QuantumInstance(backend,shots=1000)
factors_of_15=Shor(N=15,a=7,quantum_instance=quantum_instance)

In [29]:
print('factors of 15 using quantum computer are {}'.format(Shor.run(factors_of_15)))

factors of 15 using quantum computer are {'factors': [[3, 5]], 'total_counts': 65, 'successful_counts': 17}


# FINDING FACTORS OF 21

In [3]:
backend=Aer.get_backend('qasm_simulator')
quantum_instance=QuantumInstance(backend,shots=1000)
factors_of_21=Shor(N=21,a=8,quantum_instance=quantum_instance)

In [4]:
print('factors of 21 using quantum computer are {}'.format(Shor.run(factors_of_21)))

factors of 21 using quantum computer are {'factors': [[3, 7]], 'total_counts': 133, 'successful_counts': 79}


# FINDING FACTORS OF 61

In [5]:
backend=Aer.get_backend('qasm_simulator')
quantum_instance=QuantumInstance(backend,shots=1000)
factors_of_61=Shor(N=61,a=4,quantum_instance=quantum_instance)

In [6]:
print('factors of 61 using quantum computer are {}'.format(Shor.run(factors_of_61)))

factors of 61 using quantum computer are {'factors': [], 'total_counts': 222, 'successful_counts': 0}


There were no factors found for 61

# FINDING FACTORS OF 35

In [17]:
backend=Aer.get_backend('qasm_simulator')
quantum_instance=QuantumInstance(backend,shots=1000)
factors_of_35=Shor(N=35,a=8,quantum_instance=quantum_instance)

In [18]:
print('factors of 35 using quantum computer are {}'.format(Shor.run(factors_of_35)))

OverflowError: (34, 'Result too large')

overflow error due to the result being too large

# # FINDING FACTORS OF 91

In [12]:
backend=Aer.get_backend('qasm_simulator')
quantum_instance=QuantumInstance(backend,shots=1000)
factors_of_91=Shor(N=91,a=6,quantum_instance=quantum_instance)

In [8]:
print('factors of 91 using quantum computer are {}'.format(Shor.run(factors_of_91)))

AquaError: 'Circuit execution failed: ERROR:  [Experiment 0] QasmSimulator: Insufficient memory for 30-qubit circuit using "statevector" method. You could try using the "matrix_product_state" or "extended_stabilizer" method instead.'

Insufficient memory for the 30-qubit circuit used to run the experiment  while using the "statevector" method